In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competicao-tcc-ia-2025/base_teste.csv
/kaggle/input/competicao-tcc-ia-2025/base_treinamento.csv
/kaggle/input/competicao-tcc-ia-2025/exemplo_submissao.csv


In [2]:
# ======================================
# Treino Otimizado - RandomForest, LogisticRegression e XGBoost
# Ensemble por média das probabilidades
# ======================================

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd
import joblib
import numpy as np

# 1. Carregar dados
train = pd.read_csv('/kaggle/input/competicao-tcc-ia-2025/base_treinamento.csv')
test = pd.read_csv('/kaggle/input/competicao-tcc-ia-2025/base_teste.csv')

X = train.drop(columns=['target'])
y = train['target']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ======================================
# 2. Função auxiliar para treinar e salvar modelo
# ======================================
def treinar_modelo(nome, pipeline, X_train, y_train, X_val, y_val, path_modelo):
    print(f"\n=== Treinando {nome} ===")
    pipeline.fit(X_train, y_train)

    # Avaliação na validação
    y_proba = pipeline.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_proba)
    print(f"AUC ROC na validação ({nome}): {auc:.4f}")

    # Salvar modelo
    joblib.dump(pipeline, path_modelo)
    print(f"✅ Modelo salvo em: {path_modelo}")

# ======================================
# 3. Modelo 1 - RandomForest
# ======================================
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('pca', 'passthrough'),
    ('model', RandomForestClassifier(
        n_estimators=487,
        max_depth=11,
        min_samples_leaf=3,
        min_samples_split=3,
        random_state=42,
        n_jobs=-1
    ))
])

treinar_modelo("RandomForest", rf_pipeline, X_train, y_train, X_val, y_val, "modelo_randomforest.pkl")

# ======================================
# 4. Modelo 2 - LogisticRegression
# ======================================
lr_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('pca', 'passthrough'),
    ('model', LogisticRegression(
        C=0.6605,
        penalty='l2',
        solver='lbfgs',
        max_iter=1000,
        random_state=42
    ))
])

treinar_modelo("LogisticRegression", lr_pipeline, X_train, y_train, X_val, y_val, "modelo_logistic.pkl")

# ======================================
# 5. Modelo 3 - XGBoost
# ======================================
scale_pos_weight = (y_train.value_counts()[0] / y_train.value_counts()[1])

xgb_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('pca', 'passthrough'),
    ('model', XGBClassifier(
        objective='binary:logistic',
        random_state=42,
        eval_metric='auc',
        use_label_encoder=False,
        tree_method='hist',
        device='cuda',
        n_jobs=-1,
        n_estimators=369,
        max_depth=3,
        learning_rate=0.038,
        subsample=0.6746,
        colsample_bytree=0.8298,
        gamma=0,
        min_child_weight=1,
        reg_alpha=1,
        reg_lambda=5,
        scale_pos_weight=scale_pos_weight
    ))
])

treinar_modelo("XGBoost", xgb_pipeline, X_train, y_train, X_val, y_val, "modelo_xgboost.pkl")

# ======================================
# 6. Ensemble de probabilidades (média)
# ======================================
print("\n=== Gerando submissão por média das probabilidades ===")

# Carregar os 3 modelos salvos
rf_model = joblib.load("modelo_randomforest.pkl")
lr_model = joblib.load("modelo_logistic.pkl")
xgb_model = joblib.load("modelo_xgboost.pkl")

X_test = test.drop(columns=['id'])

# Obter probabilidades de cada modelo
rf_proba = rf_model.predict_proba(X_test)[:, 1]
lr_proba = lr_model.predict_proba(X_test)[:, 1]
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

# Média das probabilidades
ensemble_proba = np.mean([lr_proba, xgb_proba], axis=0)

# Criar submissão com probabilidade
submission = pd.DataFrame({
    'id': test['id'],
    'target': ensemble_proba
})

submission.to_csv('submission.csv', index=False)
print("✅ Arquivo 'submission.csv' gerado com sucesso (média das probabilidades)!")
submission.head()



=== Treinando RandomForest ===
AUC ROC na validação (RandomForest): 0.6420
✅ Modelo salvo em: modelo_randomforest.pkl

=== Treinando LogisticRegression ===
AUC ROC na validação (LogisticRegression): 0.6931
✅ Modelo salvo em: modelo_logistic.pkl

=== Treinando XGBoost ===


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [18:07:01] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


AUC ROC na validação (XGBoost): 0.6864
✅ Modelo salvo em: modelo_xgboost.pkl

=== Gerando submissão por média das probabilidades ===
✅ Arquivo 'submission.csv' gerado com sucesso (média das probabilidades)!


,id,target
0,1,0.215954
1,2,0.237482
2,3,0.328617
3,4,0.365196
4,5,0.284297
